# Session 3 · Creating Visual Designs: Images, Posters and Presentations

Generative AI › Track 3 · Innovate with Generative AI

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-03_image_prompt_builder.py`

In [1]:
"""
Session 3 · Example 1 — Build image prompts from parts, plus pixel sizes.

Image generators respond best to a prompt that names, in order:
subject -> setting -> style -> lighting -> composition -> colour palette,
plus an aspect ratio and a "negative prompt" (things to avoid).
This script builds such prompts and converts aspect ratios into pixel sizes
for common uses (posters, slides, stories).

Run:  python session-03_image_prompt_builder.py
"""
from math import gcd


def image_prompt(subject, setting, style, lighting, composition, palette,
                 ratio="1:1", avoid=()):
    parts = [subject, setting, f"{style} style", f"{lighting} lighting",
             composition, f"colour palette: {palette}"]
    prompt = ", ".join(p for p in parts if p)
    prompt += f" --ar {ratio}"
    if avoid:
        prompt += "\nNegative prompt: " + ", ".join(avoid)
    return prompt


def pixels(ratio: str, long_side: int) -> tuple:
    """Width and height for a ratio like '16:9' when the longer side is long_side px."""
    w, h = map(int, ratio.split(":"))
    if w >= h:
        return long_side, round(long_side * h / w)
    return round(long_side * w / h), long_side


basic = image_prompt("a robot reading a book", "", "cartoon", "soft", "", "pastel")
advanced = image_prompt(
    subject="a young woman coding on a laptop, smiling, wearing a dupatta",
    setting="rooftop in Karachi at dusk, city lights behind",
    style="flat vector illustration",
    lighting="warm golden-hour",
    composition="subject on the left third, empty space on the right for a headline",
    palette="deep navy, fire orange, gold",
    ratio="4:5",
    avoid=("text", "watermark", "extra fingers", "distorted face"),
)
print("BASIC PROMPT\n" + basic)
print("\nADVANCED PROMPT\n" + advanced)

print("\nASPECT RATIO -> PIXELS (long side 1920 px)")
for use, ratio in [("Slide / YouTube thumbnail", "16:9"), ("Instagram portrait post", "4:5"),
                   ("Story / Reel", "9:16"), ("Square post", "1:1"), ("A-series poster", "1:1.414")]:
    if "." in ratio:  # A-series paper is 1 : square root of 2
        w, h = round(1920 / 1.414), 1920
    else:
        w, h = pixels(ratio, 1920)
    g = gcd(w, h)
    print(f"  {use:26} {ratio:8} -> {w} x {h} px  (simplest ratio {w//g}:{h//g})")

BASIC PROMPT
a robot reading a book, cartoon style, soft lighting, colour palette: pastel --ar 1:1

ADVANCED PROMPT
a young woman coding on a laptop, smiling, wearing a dupatta, rooftop in Karachi at dusk, city lights behind, flat vector illustration style, warm golden-hour lighting, subject on the left third, empty space on the right for a headline, colour palette: deep navy, fire orange, gold --ar 4:5
Negative prompt: text, watermark, extra fingers, distorted face

ASPECT RATIO -> PIXELS (long side 1920 px)
  Slide / YouTube thumbnail  16:9     -> 1920 x 1080 px  (simplest ratio 16:9)
  Instagram portrait post    4:5      -> 1536 x 1920 px  (simplest ratio 4:5)
  Story / Reel               9:16     -> 1080 x 1920 px  (simplest ratio 9:16)
  Square post                1:1      -> 1920 x 1920 px  (simplest ratio 1:1)
  A-series poster            1:1.414  -> 1358 x 1920 px  (simplest ratio 679:960)


### Example · `session-03_contrast_checker.py`

In [2]:
"""
Session 3 · Example 2 — Is the text on your AI poster readable? (WCAG contrast)

AI tools pick attractive colours, not necessarily readable ones. The Web
Content Accessibility Guidelines (WCAG 2) define a contrast ratio:
    (L_lighter + 0.05) / (L_darker + 0.05)
where L is the "relative luminance" of each colour. Body text needs at least
4.5:1 (AA); large headings need 3:1; 7:1 is the stricter AAA level.

Run:  python session-03_contrast_checker.py
"""


def luminance(hex_colour: str) -> float:
    """Relative luminance of an sRGB colour such as '#FF6B1A'."""
    hex_colour = hex_colour.lstrip("#")
    channels = [int(hex_colour[i:i + 2], 16) / 255 for i in (0, 2, 4)]
    linear = [c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4 for c in channels]
    r, g, b = linear
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def contrast(fg: str, bg: str) -> float:
    l1, l2 = sorted((luminance(fg), luminance(bg)), reverse=True)
    return (l1 + 0.05) / (l2 + 0.05)


def verdict(ratio: float) -> str:
    if ratio >= 7:
        return "AAA (excellent)"
    if ratio >= 4.5:
        return "AA (good for body text)"
    if ratio >= 3:
        return "large headings only"
    return "FAIL - hard to read"


pairs = [
    ("Cream text on night background", "#F5EEFF", "#0A0716"),
    ("Gold text on night background",  "#FCD34D", "#0A0716"),
    ("Orange text on night background", "#FF6B1A", "#0A0716"),
    ("White text on orange button",    "#FFFFFF", "#FF6B1A"),
    ("Violet text on gold banner",     "#A78BFA", "#FCD34D"),
    ("Grey text on white page",        "#9CA3AF", "#FFFFFF"),
]
print(f"{'Pair':34} {'Ratio':>6}  Verdict")
for name, fg, bg in pairs:
    r = contrast(fg, bg)
    print(f"{name:34} {r:6.2f}  {verdict(r)}")

Pair                                Ratio  Verdict
Cream text on night background      17.59  AAA (excellent)
Gold text on night background       13.81  AAA (excellent)
Orange text on night background      6.99  AA (good for body text)
White text on orange button          2.85  FAIL - hard to read
Violet text on gold banner           1.89  FAIL - hard to read
Grey text on white page              2.54  FAIL - hard to read


### Example · `session-03_slide_outline.py`

In [3]:
"""
Session 3 · Example 3 — From a topic to a presentation outline that a slide
tool (Gamma, Copilot in PowerPoint, Canva) can turn into a deck.

Two rules keep AI decks readable:
  * one idea per slide, a headline that states the point;
  * at most 6 bullets of at most 6 words each ("6x6 rule").
The script writes a Markdown outline and checks each slide against the rules.

Run:  python session-03_slide_outline.py
"""

deck = {
    "title": "Using AI Safely in Your First Semester",
    "slides": [
        ("AI is a study partner, not a ghost-writer",
         ["Explains hard ideas", "Quizzes you before tests", "Gives feedback on drafts"]),
        ("Three prompts that make you think",
         ["Give me one hint only", "Quiz me with five questions", "Find gaps in my explanation"]),
        ("Know the integrity rules",
         ["Check each course's AI policy", "Disclose permitted AI help",
          "Never submit AI-written graded work", "Never invent references",
          "Keep your drafts as evidence", "Ask your teacher when unsure",
          "Remember that detectors are unreliable"]),
        ("Protect your data",
         ["Never paste CNIC numbers, passwords or grades into free AI tools because they may be stored"]),
    ],
}


def check_slide(bullets):
    issues = []
    if len(bullets) > 6:
        issues.append(f"{len(bullets)} bullets (max 6)")
    long = [b for b in bullets if len(b.split()) > 6]
    if long:
        issues.append(f"{len(long)} bullet(s) over 6 words")
    return issues


lines = [f"# {deck['title']}", ""]
for n, (headline, bullets) in enumerate(deck["slides"], 1):
    lines.append(f"## Slide {n}: {headline}")
    lines += [f"- {b}" for b in bullets]
    lines.append("")
    issues = check_slide(bullets)
    print(f"Slide {n}: {'OK' if not issues else 'FIX -> ' + '; '.join(issues)}")

print("\n--- Markdown outline (paste into a slide tool) ---")
print("\n".join(lines[:8]))
print("...")

Slide 1: OK
Slide 2: OK
Slide 3: FIX -> 7 bullets (max 6)
Slide 4: FIX -> 1 bullet(s) over 6 words

--- Markdown outline (paste into a slide tool) ---
# Using AI Safely in Your First Semester

## Slide 1: AI is a study partner, not a ghost-writer
- Explains hard ideas
- Quizzes you before tests
- Gives feedback on drafts

## Slide 2: Three prompts that make you think
...


### Example · `session-03_project_poster.py`

In [4]:
"""
Session 3 · Mini project (full solution) — Generate an event poster in code.

Real-life problem: image generators often misspell text inside images.
A reliable workflow is: let AI make the ARTWORK, then add the TEXT yourself
in a design tool. This script plays the "design tool" role: it lays out a
4:5 Instagram poster with a headline, details and a call to action, checks
every text colour for WCAG contrast first, and saves a PNG.

Run:  python session-03_project_poster.py
Creates: session-03_poster.png   (needs matplotlib)
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch


def luminance(h):
    h = h.lstrip("#")
    c = [int(h[i:i + 2], 16) / 255 for i in (0, 2, 4)]
    c = [x / 12.92 if x <= 0.03928 else ((x + 0.055) / 1.055) ** 2.4 for x in c]
    return 0.2126 * c[0] + 0.7152 * c[1] + 0.0722 * c[2]


def contrast(a, b):
    l1, l2 = sorted((luminance(a), luminance(b)), reverse=True)
    return (l1 + 0.05) / (l2 + 0.05)


SPEC = {
    "bg": "#0A0716",
    "headline": ("CODE THE FUTURE", "#FCD34D", 30),
    "sub": ("Inter-university AI Hackathon", "#F5EEFF", 15),
    "details": ("Sat 14 Nov  |  9 am - 9 pm  |  Main Auditorium", "#F5EEFF", 11),
    "cta": ("Register free: scan the QR at the info desk", "#0A0716", 11),
    "cta_bg": "#FF6B1A",
}

# 1) Quality gate: every text/background pair must pass before we draw.
checks = [("headline", SPEC["headline"][1], SPEC["bg"], 3.0),
          ("sub", SPEC["sub"][1], SPEC["bg"], 4.5),
          ("details", SPEC["details"][1], SPEC["bg"], 4.5),
          ("cta", SPEC["cta"][1], SPEC["cta_bg"], 4.5)]
for name, fg, bg, need in checks:
    r = contrast(fg, bg)
    print(f"{name:9} contrast {r:5.2f} (needs {need}) -> {'PASS' if r >= need else 'FAIL'}")
    assert r >= need, f"{name} is not readable; pick another colour"

# 2) Draw the poster: 4:5 portrait, 4 x 5 inches at 120 dpi = 480 x 600 px.
fig = plt.figure(figsize=(4, 5), dpi=120)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, 4); ax.set_ylim(0, 5); ax.axis("off")
fig.patch.set_facecolor(SPEC["bg"])
# "artwork" placeholder: glowing circles where AI-generated art would sit
for r, a in [(1.6, 0.10), (1.15, 0.18), (0.75, 0.35)]:
    ax.add_patch(Circle((2, 3.35), r, color="#FF6B1A", alpha=a))
ax.add_patch(Circle((2, 3.35), 0.45, color="#F43F5E", alpha=0.8))
text, colour, size = SPEC["headline"]
ax.text(2, 1.95, text, ha="center", color=colour, fontsize=size * 0.8, weight="bold")
for key, y in (("sub", 1.55), ("details", 1.2)):
    t, c, s = SPEC[key]
    ax.text(2, y, t, ha="center", color=c, fontsize=s * 0.8)
ax.add_patch(FancyBboxPatch((0.35, 0.45), 3.3, 0.45, boxstyle="round,pad=0.05", color=SPEC["cta_bg"]))
t, c, s = SPEC["cta"]
ax.text(2, 0.62, t, ha="center", color=c, fontsize=s * 0.75, weight="bold")
fig.savefig("session-03_poster.png", facecolor=SPEC["bg"])
w, h = fig.get_size_inches() * fig.dpi
print(f"Saved poster -> session-03_poster.png ({int(w)} x {int(h)} px, ratio 4:5)")

headline  contrast 13.81 (needs 3.0) -> PASS
sub       contrast 17.59 (needs 4.5) -> PASS
details   contrast 17.59 (needs 4.5) -> PASS
cta       contrast  6.99 (needs 4.5) -> PASS
Saved poster -> session-03_poster.png (480 x 600 px, ratio 4:5)
